In [2]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Folder utama praktikum
BASE_DIR = Path("/content/drive/MyDrive/BigData/Praktikum5")

print("Folder penyimpanan:", BASE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder penyimpanan: /content/drive/MyDrive/BigData/Praktikum5


# H-2: Persiapan Unduh Data dan Jalankan Portal Lokal

In [3]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
  urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas  :", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom   :", df_sumber.shape)
print("Kolom          :", list(df_sumber.columns))

import http.server, urllib.parse, html

HALAMAN_UKURAN = 100        # judul per halaman

def jalankan_portal(df, port=8000):
  """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
  baris = df.to_dict("records")
  n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

  class Handler(http.server.BaseHTTPRequestHandler):
    def log_message(self, *args):       # matikan log
        pass
    def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
        data = teks.encode("utf-8")
        self.send_response(kode)
        self.send_header("Content-Type", tipe)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)
    def do_GET(self):
        u = urllib.parse.urlparse(self.path)
        if u.path == "/robots.txt":
            self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
        elif u.path == "/arsip-privat/":
            self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
        elif u.path == "/berita":
            q = urllib.parse.parse_qs(u.query)
            hal = int(q.get("hal", ["1"])[0])
            if not 1 <= hal <= n_hal:
                return self._kirim(404, "<p>Halaman tidak ada</p>")
            potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
            isi = []
            for r in potong:
                iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                isi.append(
                    '<article class="item">'
                    f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                    f'<span class="kategori">{html.escape(r["category"])}</span>'
                    f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                    '</article>')
            self._kirim(200, f"<html><body><h1>Portal Contoh<h1>{''.join(isi)}"
                             f'<nav data-total="{n_hal}"></nav></body></html>')
        else:
            self._kirim(404, "<p>TIdak ditemukan</p>")

  server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
  threading.Thread(target=server.serve_forever, daemon=True).start()
  return server, n_hal

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Ukuran berkas  : 16.1 MB
Baris, kolom   : (91017, 4)
Kolom          : ['date', 'url', 'title', 'category']
Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


# H-3: Crawling Portal Lokal: Latihan Aman

In [4]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


In [5]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
  """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
  rp = baca_robots(base)
  sesi = requests.Session()
  sesi.headers["User-Agent"] = ua
  for h in halaman:
      url = f"{base}/berita?hal={h}"
      if not rp.can_fetch(ua, url):
          print("Dilewati (robots.txt):", url)
          continue
      for percobaan in range(3):
          try:
              r = sesi.get(url, timeout=10)
              r.raise_for_status()
              break
          except requests.RequestException as e:
              time.sleep(0.5 * (percobaan + 1))
      else:
          print("Gagal setelah 3 percobaan:", url)
          continue
      yield from parse_halaman(r.text)
      time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
  print(c)

Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


# H-4: Crawling Berita Sungguhan dari Sitemap detik.com

In [6]:
import re
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(
    r"https?://([\w-]+)\.detik\.com/.+?/d-(\d+)/"
)

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []

    for u in sup.find_all("url"):
        loc = u.find("loc")
        judul = u.find("title")
        waktu = u.find("publication_date")

        if loc is None or judul is None:
            continue

        # Membersihkan URL dari whitespace dan newline
        url = "".join(loc.get_text().split())

        # Mencocokkan pola URL
        m = POLA_ARTIKEL.search(url)

        # Mengambil kata kunci
        kunci = u.find("keywords")

        # Mengambil kategori dari subdomain
        parsed_url = urlparse(url)
        kategori = parsed_url.netloc.split(".")[0]

        hasil.append({
            "judul": re.sub(
                r"\s+", " ", judul.get_text(strip=True)
            ),
            "url": url,
            "kategori": kategori,
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True) if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })

    return hasil

In [7]:
# Contoh XML: bentuk disalin dari sitemap berita asli (tiga entri hasil uji dosen, 29 Okt 2026).
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa
diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>

<news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan
DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
   <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari
lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak
Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat
banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan
Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) ==
parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29T12:52:00+00:00'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29T12:50:00+00:00'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29T12:44:00+00:00'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


In [8]:
import urllib.robotparser

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: amelianiisrina@gmail.com)"
DELAY = 2.0                 # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7        # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20            # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}
def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content            # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

In [9]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul   :", len(cek))
    print("Judul kosong      :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat      :", cek["url"].duplicated().sum())
    print("Tanggal kosong    :", cek["tanggal"].isna().sum())
    print("Kategori          :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul   : 20
Judul kosong      : 0
URL duplikat      : 6
Tanggal kosong    : 0
Kategori          : {'news': 7, '': 7, 'sport': 6}
                                                                    judul kategori                   tanggal
                      Heboh Temuan 1,2 Juta Pil Tramadol di Gunung Sindur     news 2026-11-07T21:56:00+07:00
          Tenant Hall di Stasiun Gambir Kebakaran, Damkar Meluncur ke TKP     news 2026-11-07T21:47:00+07:00
MPR Ajak Guru Jadi Garda Terdepan Perkuat Nilai Kebangsaan di Era Digital     news 2026-11-07T21:45:00+07:00


# H-5: Channel ala Flume ke lapisan_mentah

In [10]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/BigData/Praktikum5")

RAW = BASE_DIR / "lapisan_mentah"
RAW.mkdir(parents=True, exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)                       # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []
    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0
print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink  :", f"{statistik['keluar']:,}")
print("File batch          :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3],
"...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 65.8 detik untuk 911 halaman


# H-6: Quality, Cleaning, dan Parquet Berpartisi

In [11]:
import pandas as pd
import re

mentah = pd.concat(
    [pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))],
    ignore_index=True
)

print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {
    "finance", "food", "health", "hot", "inet",
    "news", "oto", "sport", "travel"
}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""

    jml_kata = d["judul"].fillna("").str.split().str.len()

    return {
        "judul tidak kosong":
            d["judul"].fillna("").str.strip() != "",

        "judul 3-30 kata":
            jml_kata.between(3, 30),

        "kategori dikenal":
            d["kategori"].isin(KATEGORI_DIKENAL),

        "tanggal valid":
            pd.to_datetime(
                d["tanggal"],
                format="%Y-%m-%d",
                errors="coerce"
            ).notna(),

        "url artikel valid":
            d["url"].fillna("").str.match(
                r"^https?://[\w-]+\.detik\.com/.+d-\d+/"
            ),

        "judul tidak duplikat":
            ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(
        f"{nama:<24} gagal: {(~lulus).sum():>6,} "
        f"({(~lulus).mean()*100:5.2f}%)"
    )


Baris mentah: 91,017
judul tidak kosong       gagal:      0 ( 0.00%)
judul 3-30 kata          gagal:     78 ( 0.09%)
kategori dikenal         gagal:      0 ( 0.00%)
tanggal valid            gagal:      0 ( 0.00%)
url artikel valid        gagal:  5,640 ( 6.20%)
judul tidak duplikat     gagal:    235 ( 0.26%)


In [12]:
# Membuat salinan dataset mentah
bersih = mentah.copy()

# Membersihkan whitespace pada judul
bersih["judul"] = (
    bersih["judul"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Menghapus judul duplikat
bersih = bersih.drop_duplicates(
    subset="judul",
    keep="first"
)

# Mempertahankan judul dengan 3-30 kata
bersih = bersih[
    bersih["judul"].str.split().str.len().between(3, 30)
].copy()

# Mengubah tanggal menjadi format datetime
bersih["tanggal"] = pd.to_datetime(
    bersih["tanggal"],
    format="%Y-%m-%d"
)

# Membuat kolom bulan
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")

print("Baris mentah :", f"{len(mentah):,}")
print("Baris bersih :", f"{len(bersih):,}")
print("Baris terhapus:", f"{len(mentah) - len(bersih):,}")

bersih["id"] = (
    bersih["url"]
    .str.extract(r"/d-(\d+)/")[0]
    .astype("Int64")
)


Baris mentah : 91,017
Baris bersih : 90,818
Baris terhapus: 199


In [13]:
import shutil
import pyarrow.dataset as ds
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/BigData/Praktikum5")
BASE_DIR.mkdir(parents=True, exist_ok=True)

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran


# BAGIAN KUNCI: bandingkan dua pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> "f"{n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")


# Menyimpan hasil terkurasi ke Google Drive
TERKURASI = BASE_DIR / "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], str(TERKURASI))
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")


dataset = ds.dataset(str(TERKURASI), format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori = finance")


partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke /content/drive/MyDrive/BigData/Praktikum5/lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


# H-7: Query DuckDB dan Bentuk Dokumen

In [14]:
import duckdb
import json
con = duckdb.connect()
con.sql(f"""CREATE OR REPLACE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet',
        hive_partitioning=true)""")

print(con.sql("""SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah
DESC""").df().to_string(index=False))
print()

print(con.sql("""
      SELECT bulan, COUNT(*) AS judul_corona
      FROM berita WHERE lower(judul) LIKE '%corona%'
      GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))


kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


# H-8: Mini MapReduce: Kasus Kantin

In [15]:
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],      # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],      # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"], # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner:", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


# H-9: Word Count Terdistribusi: Partisi sebagai Input Split

In [16]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")
print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")
top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723

10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317

Sama dengan Counter biasa: True


# H-10: Combiner dan Jumlah Data yang Berpindah

In [17]:
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")

for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for
x in h):>20,}")
    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for
x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.698              825,601
1 proses, dengan combiner         0.674               74,009
2 proses, tanpa combiner          2.341              825,601
2 proses, dengan combiner         1.274               74,009

Hasil dengan combiner sama persis: True


# H-11: Data Skew dan Beban Reducer

In [18]:
R = 4        # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22}", " ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

tanpa combiner         R0=240,836 (29.2%) R1=170,066 (20.6%) R2=190,727 (23.1%) R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%) R1= 18,267 (24.7%) R2= 18,530 (25.0%) R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    4 reducer: terberat 29.17% | bagian adil 25.00%
   16 reducer: terberat  8.82% | bagian adil  6.25%
   64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


# H-12: Hadoop Streaming

In [19]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
           print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')
print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


In [20]:
!head -n 3 masukan.tsv | python3 mapper.py

!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1

real	0m2.055s
user	0m2.240s
sys	0m0.083s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785


In [21]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


In [22]:
%%bash
sudo apt-get install -y openjdk-11-jdk-headless
wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
tar -xzf hadoop-3.3.6.tar.gz
export HADOOP_HOME=$PWD/hadoop-3.3.6
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64

rm -rf keluaran_hadoop

$HADOOP_HOME/bin/hadoop jar \
  $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" \
  -reducer "python3 reducer.py" \
  -input masukan.tsv \
  -output keluaran_hadoop

ls keluaran_hadoop

diff <(LC_ALL=C sort hasil_pipe.txt) keluaran_hadoop/part-00000 && echo "SAMA"

Reading package lists...
Building dependency tree...
Reading state information...
openjdk-11-jdk-headless is already the newest version (11.0.32.1+1-1ubuntu1~24.04).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.
part-00000
_SUCCESS
SAMA


2026-11-07 15:21:51,066 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-11-07 15:21:51,323 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-11-07 15:21:51,323 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-11-07 15:21:51,356 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-11-07 15:21:51,764 INFO mapred.FileInputFormat: Total input files to process : 1
2026-11-07 15:21:51,809 INFO mapreduce.JobSubmitter: number of splits:1
2026-11-07 15:21:52,286 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1622688666_0001
2026-11-07 15:21:52,286 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-07 15:21:52,818 INFO mapred.LocalDistributedCacheManager: Localized file:/content/mapper.py as file:/tmp/hadoop-root/mapred/local/job_local1622688666_0001_29eac9dc-341e-4384-a478-000db126bd21/mapper.py
2026-11-07 15:21:52,854 INFO mapred.LocalDistributedCacheMa

# H-13: Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman

In [23]:
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))
for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

import sqlite3
SIAP = BASE_DIR / "siap_pakai"
SIAP.mkdir(parents=True, exist_ok=True)
baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print("Baris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
                    WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)
Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


# Latihan

1. Stopword domain. Tambahkan 5 kata ke STOPWORD yang menurut Anda mengganggu hasil, lalu
jalankan ulang H-13. Catat kata yang masuk dan perubahan lima kata teratas tiap kategori.

In [24]:
STOPWORD_LATIHAN = STOPWORD.copy()

STOPWORD_LATIHAN.update(["rp", "ri", "the", "ini", "yang"])

print("Jumlah stopword awal:", len(STOPWORD))
print("Jumlah stopword latihan:", len(STOPWORD_LATIHAN))
print("Stopword tambahan:", ["rp", "ri", "the", "ini", "yang"])

Jumlah stopword awal: 48
Jumlah stopword latihan: 51
Stopword tambahan: ['rp', 'ri', 'the', 'ini', 'yang']


In [25]:
def mapper_kategori_latihan(path):
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]

    c = collections.Counter()

    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD_LATIHAN:
                c[(kategori, kata)] += 1

    return c

In [26]:
gabung_latihan = collections.Counter()

for c in map(mapper_kategori_latihan, splits):
    gabung_latihan.update(c)

per_kategori_latihan = collections.defaultdict(list)

for (kat, kata), n in gabung_latihan.items():
    per_kategori_latihan[kat].append((n, kata))

print("5 kata teratas setelah penambahan stopword:\n")

for kat in sorted(per_kategori_latihan):
    top5 = sorted(per_kategori_latihan[kat], reverse=True)[:5]

    print(
        f"{kat:<8}",
        ", ".join(f"{kata} ({jumlah:,})" for jumlah, kata in top5)
    )

5 kata teratas setelah penambahan stopword:

finance  corona (2,470), harga (701), jokowi (642), mau (562), ekonomi (536)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), film (489)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), indonesia (256), honda (255)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)


In [27]:
print("PERBANDINGAN H-13 SEBELUM DAN SESUDAH LATIHAN")
print("Stopword tambahan: rp, ri, the, ini, yang\n")

for kat in sorted(per_kategori_latihan):
    top5 = sorted(per_kategori_latihan[kat], reverse=True)[:5]

    print(f"{kat}:")
    print("Sesudah:", ", ".join(
        f"{kata} ({jumlah:,})" for jumlah, kata in top5
    ))
    print()

PERBANDINGAN H-13 SEBELUM DAN SESUDAH LATIHAN
Stopword tambahan: rp, ri, the, ini, yang

finance:
Sesudah: corona (2,470), harga (701), jokowi (642), mau (562), ekonomi (536)

food:
Sesudah: makanan (699), makan (539), foto (458), galeri (438), enak (358)

health:
Sesudah: corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)

hot:
Sesudah: corona (1,658), anak (793), rumah (561), meninggal (545), film (489)

inet:
Sesudah: corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)

news:
Sesudah: corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)

oto:
Sesudah: mobil (1,205), corona (576), motor (432), indonesia (256), honda (255)

sport:
Sesudah: corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)

travel:
Sesudah: corona (1,023), wisata (525), foto (333), dunia (299), turis (284)



Penambahan 5 stopword, yaitu 'rp', 'ri', 'the', 'ini', dan 'yanhg', menyebabkan perubahan pada daftar lima kata teratas di beberapa kategori. Kata-kata tersebut tidak lagi muncul dalam hasil karena sudah diabaikan saat perhitungan. Contohnya, pada kategori finance, kata 'rp' dan 'ri' tergantikan oleh 'mau' dan 'ekonomi'. Pada kategori hot, kata 'the' tergantikan oleh 'film'. Sementara itu, kategori food, health, inet, news, sport, dan travel tidak mengalami perubahan pada lima kata teratasnya. Hal ini menunjukkan bahwa penambahan stopword dapat membantu menampilkan kata-kata yang lebih menggambarkan isi berita.

2. Per bulan. Ubah key di H-13 menjadi (bulan, kata). Kata apa yang berubah dari Januari ke Mei?

In [28]:
def mapper_bulan(path):
    df_ = pd.read_parquet(path, columns=["judul", "tanggal"])

    c = collections.Counter()

    for _, row in df_.iterrows():
        bulan = pd.to_datetime(row["tanggal"]).strftime("%Y-%m")

        for kata in tokenisasi(row["judul"]):
            if kata not in STOPWORD:
                c[(bulan, kata)] += 1

    return c

In [29]:
gabung_bulan = collections.Counter()

for c in map(mapper_bulan, splits):
    gabung_bulan.update(c)

print("Jumlah pasangan:", sum(gabung_bulan.values()))
print("Jumlah kombinasi bulan dan kata:", len(gabung_bulan))

Jumlah pasangan: 709926
Jumlah kombinasi bulan dan kata: 85925


In [30]:
per_bulan = collections.defaultdict(list)

for (bulan, kata), jumlah in gabung_bulan.items():
    per_bulan[bulan].append((jumlah, kata))

for bulan in ["2020-01", "2020-05"]:
    print(f"\n10 kata teratas bulan {bulan}:")

    top10 = sorted(per_bulan[bulan], reverse=True)[:10]

    for jumlah, kata in top10:
        print(f"{kata:<12} {jumlah:>7,}")


10 kata teratas bulan 2020-01:
banjir         1,220
rp               616
polisi           554
jokowi           516
tahun            493
baru             475
mobil            428
minta            427
kpk              415
indonesia        410

10 kata teratas bulan 2020-05:
corona         2,935
kasus            640
positif          628
pandemi          596
warga            565
lebaran          562
rp               558
psbb             522
covid            522
polisi           517


In [31]:
top_januari = {
    kata for jumlah, kata in
    sorted(per_bulan["2020-01"], reverse=True)[:10]
}

top_mei = {
    kata for jumlah, kata in
    sorted(per_bulan["2020-05"], reverse=True)[:10]
}

print("Kata yang hanya muncul di 10 besar Januari:")
print(top_januari - top_mei)

print("\nKata yang hanya muncul di 10 besar Mei:")
print(top_mei - top_januari)

Kata yang hanya muncul di 10 besar Januari:
{'banjir', 'tahun', 'jokowi', 'minta', 'baru', 'indonesia', 'mobil', 'kpk'}

Kata yang hanya muncul di 10 besar Mei:
{'corona', 'pandemi', 'kasus', 'covid', 'warga', 'positif', 'lebaran', 'psbb'}


Berdasarkan hasil analisis, terdapat perubahan kata yang sering muncul dari Januari hingga Mei 2020. Pada Januari, kata yang paling banyak muncul adalah 'banjir' sebanyak 1.220 kali, diikuti oleh 'polisi', 'jokowi', dan 'kpk'. Sementara itu, pada Mei, kata 'corona' menjadi yang paling banyak muncul dengan jumlah 2.935 kali, diikuti oleh 'kasus', 'positif', 'pandemi', dan 'covid'.

Beberapa kata yang masuk dalam 10 besar pada Januari tetapi tidak lagi muncul pada Mei adalah 'banjir', 'jokowi', 'kpk', dan 'mobil'. Sebaliknya, kata seperti 'corona', 'pandemi', 'covid', 'psbb', dan 'positif' mulai masuk ke dalam 10 besar pada Mei.

Hal ini menunjukkan adanya perubahan fokus pemberitaan. Pada Januari, berita membahas berbagai topik, sedangkan pada Mei pemberitaan lebih banyak berfokus pada pandemi COVID-19.

3. Bigram. Ubah mapper agar menghasilkan pasangan kata berurutan, misalnya "virus corona". Berapa pasangan yang dihasilkan, dan berapa setelah combiner?

In [32]:
def mapper_bigram(path):
    df_ = pd.read_parquet(path, columns=["judul"])

    c = collections.Counter()
    total = 0

    for judul in df_["judul"]:
        kata = tokenisasi(judul)

        # Membuat pasangan dua kata berurutan
        bigram = list(zip(kata, kata[1:]))

        total += len(bigram)
        c.update(" ".join(pasangan) for pasangan in bigram)

    return c, total

In [33]:
hasil_mapper_bigram = [
    mapper_bigram(p) for p in splits
]

# Jumlah pasangan sebelum combiner
total_sebelum = sum(total for c, total in hasil_mapper_bigram)

# Jumlah pasangan setelah combiner
total_sesudah = sum(len(c) for c, total in hasil_mapper_bigram)

# Menggabungkan hasil seluruh mapper
hasil_bigram = collections.Counter()

for c, total in hasil_mapper_bigram:
    hasil_bigram.update(c)

print("Pasangan sebelum combiner:", f"{total_sebelum:,}")
print("Pasangan setelah combiner:", f"{total_sesudah:,}")
print("Bigram unik:", f"{len(hasil_bigram):,}")

pengurangan = (1 - total_sesudah / total_sebelum) * 100

print(f"Pengurangan pasangan: {pengurangan:.2f}%")

Pasangan sebelum combiner: 734,783
Pasangan setelah combiner: 411,217
Bigram unik: 349,114
Pengurangan pasangan: 44.04%


In [34]:
print("\n10 bigram paling sering muncul:")

for bigram, jumlah in hasil_bigram.most_common(10):
    print(f"{bigram:<25} {jumlah:>7,}")


10 bigram paling sering muncul:
virus corona                2,915
corona di                   2,178
positif corona              1,804
di tengah                   1,258
pandemi corona              1,075
di rumah                    1,047
new normal                    917
di indonesia                  871
rp juta                       700
hari ini                      627


Berdasarkan hasil percobaan, mapper menghasilkan 734.783 pasangan kata sebelum menggunakan combiner. Setelah combiner diterapkan, jumlah pasangan yang dikirim berkurang menjadi 411.217 pasangan atau berkurang sebear 44,04%. Jumlah bigram unik yang dihasilkan adalah 349.114 pasangan.

Bigram yang paling sering muncul adalah 'virus corona' sebanyak 2.915 kali, diikuti 'corona di' sebanyak 2.178 kali, dan 'positif corona' sebanyak 1.804 kali. Hasil ini menunjukkan bahwa topik pemberitaan banyak berkaitan dengan virus corona dan dampaknya.

Combiner membantu mengurangi jumlah data yang harus dikirim karena pasangan kata yang sama digabungkan terlebih dahulu pada masing-masing mapper. Dengan begitu, proses pengolahan data menjadi lebih efisien.

4. Jumlah reducer. Jalankan H-11 dengan R = 2, 8, dan 32. Buat tabel beban reducer terberat dibanding bagian yang adil.

In [35]:
def hitung_beban(pasangan_per_mapper, r):
    b = [0] * r

    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1

    return b

In [36]:
hasil_reducer = []

for r in (2, 8, 32):
    b = hitung_beban(hasil_map, r)

    total = sum(b)
    terberat = max(b)
    reducer_terberat = b.index(terberat)

    persen_terberat = terberat / total * 100
    bagian_adil = 100 / r

    hasil_reducer.append({
        "Jumlah Reducer": r,
        "Reducer Terberat": f"R{reducer_terberat}",
        "Beban Terberat (%)": round(persen_terberat, 2),
        "Bagian Adil (%)": round(bagian_adil, 2)
    })

tabel_reducer = pd.DataFrame(hasil_reducer)

print(tabel_reducer.to_string(index=False))

 Jumlah Reducer Reducer Terberat  Beban Terberat (%)  Bagian Adil (%)
              2               R0               52.27            50.00
              8               R0               16.83            12.50
             32               R0                5.82             3.12


In [37]:
for r in (2, 8, 32):
    b = hitung_beban(hasil_map, r)
    total = sum(b)

    print(f"\nR = {r}")
    for i, jumlah in enumerate(b):
        print(
            f"Reducer R{i}: {jumlah:,} pasangan "
            f"({jumlah / total * 100:.2f}%)"
        )


R = 2
Reducer R0: 431,563 pasangan (52.27%)
Reducer R1: 394,038 pasangan (47.73%)

R = 8
Reducer R0: 138,978 pasangan (16.83%)
Reducer R1: 77,144 pasangan (9.34%)
Reducer R2: 82,045 pasangan (9.94%)
Reducer R3: 109,178 pasangan (13.22%)
Reducer R4: 101,858 pasangan (12.34%)
Reducer R5: 92,922 pasangan (11.26%)
Reducer R6: 108,682 pasangan (13.16%)
Reducer R7: 114,794 pasangan (13.90%)

R = 32
Reducer R0: 48,073 pasangan (5.82%)
Reducer R1: 17,153 pasangan (2.08%)
Reducer R2: 19,911 pasangan (2.41%)
Reducer R3: 31,279 pasangan (3.79%)
Reducer R4: 30,981 pasangan (3.75%)
Reducer R5: 20,621 pasangan (2.50%)
Reducer R6: 40,913 pasangan (4.96%)
Reducer R7: 18,083 pasangan (2.19%)
Reducer R8: 40,656 pasangan (4.92%)
Reducer R9: 21,146 pasangan (2.56%)
Reducer R10: 18,779 pasangan (2.27%)
Reducer R11: 24,102 pasangan (2.92%)
Reducer R12: 21,112 pasangan (2.56%)
Reducer R13: 24,563 pasangan (2.98%)
Reducer R14: 22,946 pasangan (2.78%)
Reducer R15: 35,697 pasangan (4.32%)
Reducer R16: 24,718 p

In [38]:
hasil_reducer = []

for r in (2, 8, 32):
    b = hitung_beban(hasil_map, r)
    total = sum(b)

    hasil_reducer.append({
        "Jumlah Reducer": r,
        "Beban Terberat (%)": round(max(b) / total * 100, 2),
        "Bagian Adil (%)": round(100 / r, 2)
    })

tabel_reducer = pd.DataFrame(hasil_reducer)
display(tabel_reducer)

,Jumlah Reducer,Beban Terberat (%),Bagian Adil (%)
0,2,52.27,50.00
1,8,16.83,12.50
2,32,5.82,3.12


Berdasarkan hasil percobaan, semakin banyak jumlah reducer, semakin kecil beban yang diterima oleh reducer terberat. Pada 2 reducer, beban terberat mencapai 52,27% sedangkan pada 8 reducer turun menjadi 16,83% dan pada 32 reducer menjadi 5,82%.

Namun, beban tersebut masih lebih besar dibandingkan bagian idealnya. Hal ini menunjukkan bahwa pembagian data belum sepenuhnya merata. Jadi, penambahan jumlah reducer dapat memantu menyebarkan beban kerja, tetapi tidak selalu membuat pembagiannya sama rata.

5. Crawl ulang. Tambahkan jeda 0,05 detik pada crawl_iter untuk 50 halaman pertama. Berapa
waktunya, dan berapa lama bila 911 halaman memakai jeda 1 detik?

In [39]:
import time

t0 = time.perf_counter()

hasil_latihan = list(
    crawl_iter(BASE, range(1, 51), delay=0.05)
)

waktu = time.perf_counter() - t0

print("Jumlah halaman     : 50")
print("Jumlah berita      :", len(hasil_latihan))
print("Waktu crawling     :", round(waktu, 2), "detik")

# Estimasi 911 halaman dengan jeda 1 detik
estimasi = (waktu / 50 * 911) + (911 - 50) * (1 - 0.05)

print("\nEstimasi 911 halaman dengan jeda 1 detik:")
print("Waktu              :", round(estimasi, 2), "detik")
print("Dalam menit        :", round(estimasi / 60, 2), "menit")

Jumlah halaman     : 50
Jumlah berita      : 5000
Waktu crawling     : 5.53 detik

Estimasi 911 halaman dengan jeda 1 detik:
Waktu              : 918.7 detik
Dalam menit        : 15.31 menit


Berdasarkan percobaan, proses crawling terhadap 50 halaman berhasil mendapatkan 5.000 berita dengan waktu 6,41 detik menggunakan jeda 0,05 detik.

Jika proses crawling dilakukan terhadap 911 halaman dengan jeda 1 detik, waktu yang diperkirakan adalah 934,73 detik atau sekitar 15,58 menit.

Hasil ini menunjukkan bahwa penambahan jeda waktu dapat membuat proses crawling menjadi lebih lama. Namun, jeda diperlukan agar proses pengambilan data tidak terlalu sering mengirim permintaan ke server. Dengan demikian, penggunaan jeda membantu proses crawling berjalan lebih teratur dan mengurangi beban pada server.

6.  Sitemap lain. Ganti satu alamat di SITEMAP_DIPILIH dengan sitemap_news dari bagian lain yang tercantum di sitemap.xml. Kategori apa yang muncul? Terapkan aturan_kualitas dari H-6 pada 20 judul hasil crawl dan laporkan hasilnya.

In [40]:
import xml.etree.ElementTree as ET
url_index = "https://www.detik.com/sitemap.xml"

sesi = requests.Session()
sesi.headers["User-Agent"] = UA_DETIK

xml_index = ambil_xml(url_index, sesi)
root = ET.fromstring(xml_index)

daftar_url = [
    ele.text.strip()
    for ele in root.iter()
    if ele.tag.endswith("loc") and ele.text
]

sitemap_news = [
    url for url in daftar_url
    if "sitemap_news" in url
]

for url in sitemap_news:
    print(url)

https://news.detik.com/suara-pembaca/sitemap_news.xml
https://news.detik.com/berita/sitemap_news.xml
https://news.detik.com/internasional/sitemap_news.xml
https://news.detik.com/bbc-world/sitemap_news.xml
https://news.detik.com/abc-australia/sitemap_news.xml
https://news.detik.com/dw/sitemap_news.xml
https://news.detik.com/kolom/sitemap_news.xml
https://news.detik.com/prokontra/sitemap_news.xml
https://news.detik.com/pemilu/sitemap_news.xml
https://news.detik.com/pilkada/sitemap_news.xml
https://news.detik.com/melindungi-tuah-marwah/sitemap_news.xml
https://news.detik.com/bangun-indonesia/sitemap_news.xml
https://news.detik.com/nyago-bumi-sriwijaya/sitemap_news.xml
https://inet.detik.com/news/sitemap_news.xml
https://inet.detik.com/business/sitemap_news.xml
https://inet.detik.com/security/sitemap_news.xml
https://inet.detik.com/telecommunication/sitemap_news.xml
https://inet.detik.com/law-and-policy/sitemap_news.xml
https://inet.detik.com/cyberlife/sitemap_news.xml
https://inet.detik.c

In [41]:
SITEMAP_LATIHAN = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://news.detik.com/bbc-world/sitemap_news.xml",
]

BATAS_PER_BAGIAN = 7
BATAS_TOTAL = 20
DELAY = 2.0

In [42]:
def crawl_latihan():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK

    hasil = []

    for sm in SITEMAP_LATIHAN:
        if len(hasil) >= BATAS_TOTAL:
            break

        data = parse_sitemap_news(
            ambil_xml(sm, sesi, delay=DELAY)
        )

        # Menambahkan kategori berdasarkan bagian sitemap
        for berita in data[:BATAS_PER_BAGIAN]:
            hasil.append(berita)

    return hasil[:BATAS_TOTAL]

In [43]:
hasil_latihan = crawl_latihan()

df_latihan = pd.DataFrame(hasil_latihan)

print("Jumlah judul:", len(df_latihan))

display(
    df_latihan[
        ["judul", "url", "kategori", "tanggal"]
    ]
)

Jumlah judul: 20


,judul,url,kategori,tanggal
0,"Heboh Temuan 1,2 Juta Pil Tramadol di Gunung S...",https://news.detik.com/berita/d-8743070/heboh-...,news,2026-11-07T21:56:00+07:00
1,"Tenant Hall di Stasiun Gambir Kebakaran, Damka...",https://news.detik.com/berita/d-8743061/tenant...,news,2026-11-07T21:47:00+07:00
2,MPR Ajak Guru Jadi Garda Terdepan Perkuat Nila...,https://news.detik.com/berita/d-8743059/mpr-aj...,news,2026-11-07T21:45:00+07:00
3,Istri di Jakbar Dianiaya hingga Disundut Rokok...,https://news.detik.com/berita/d-8743055/istri-...,news,2026-11-07T21:42:00+07:00
4,Damkar Evakuasi Cincin Tersangkut di Kemaluan ...,https://news.detik.com/berita/d-8742997/damkar...,news,2026-11-07T20:47:00+07:00
5,"Seskab Teddy Pamer Fasilitas Sekolah Rakyat, A...",https://news.detik.com/berita/d-8742990/seskab...,news,2026-11-07T20:31:00+07:00
6,Om Tikam Keponakan hingga Kritis di Maros Usai...,https://news.detik.com/berita/d-8742985/om-tik...,news,2026-11-07T20:27:00+07:00
7,Nokia Menatap Masa Depan Tanpa Bisnis Ponsel,,,2014-06-24T18:09:00+07:00
8,"Telkomsel Siap Komersilkan LTE, Asalkan...",,,2014-06-24T16:24:00+07:00
9,"T-Zone, 'BTS Mini' untuk Area Lemah Sinyal",,,2014-06-24T14:48:00+07:00


In [44]:
KATEGORI_DIKENAL = {
    "finance", "food", "health", "hot", "inet",
    "news", "oto", "sport", "travel"
}

def aturan_kualitas_latihan(d):

    jml_kata = d["judul"].fillna("").str.split().str.len()

    return {
        "judul tidak kosong":
            d["judul"].fillna("").str.strip() != "",

        "judul 3-30 kata":
            jml_kata.between(3, 30),

        "kategori dikenal":
            d["kategori"].isin(KATEGORI_DIKENAL),

        "tanggal valid":
            pd.to_datetime(
                d["tanggal"],
                errors="coerce"
            ).notna(),

        "url artikel valid":
            d["url"].fillna("").str.match(
                r"^https?://[\w-]+\.detik\.com/.+d-\d+/"
            ),

        "judul tidak duplikat":
            ~d["judul"].duplicated(keep=False),
    }


hasil_aturan = aturan_kualitas_latihan(df_latihan)

print("HASIL PEMERIKSAAN KUALITAS\n")

for nama, lulus in hasil_aturan.items():
    gagal = (~lulus).sum()
    persen = gagal / len(df_latihan) * 100

    print(
        f"{nama:<24} gagal: {gagal:>3} "
        f"({persen:5.2f}%)"
    )

HASIL PEMERIKSAAN KUALITAS

judul tidak kosong       gagal:   0 ( 0.00%)
judul 3-30 kata          gagal:   0 ( 0.00%)
kategori dikenal         gagal:   7 (35.00%)
tanggal valid            gagal:   0 ( 0.00%)
url artikel valid        gagal:   7 (35.00%)
judul tidak duplikat     gagal:   0 ( 0.00%)


In [45]:
semua_lulus = pd.DataFrame(hasil_aturan).all(axis=1)

print("\nRINGKASAN")
print("Total judul             :", len(df_latihan))
print("Judul lolos semua aturan:", semua_lulus.sum())
print("Judul gagal             :", (~semua_lulus).sum())

display(
    df_latihan[semua_lulus][
        ["judul", "kategori", "tanggal", "url"]
    ]
)


RINGKASAN
Total judul             : 20
Judul lolos semua aturan: 13
Judul gagal             : 7


,judul,kategori,tanggal,url
0,"Heboh Temuan 1,2 Juta Pil Tramadol di Gunung S...",news,2026-11-07T21:56:00+07:00,https://news.detik.com/berita/d-8743070/heboh-...
1,"Tenant Hall di Stasiun Gambir Kebakaran, Damka...",news,2026-11-07T21:47:00+07:00,https://news.detik.com/berita/d-8743061/tenant...
2,MPR Ajak Guru Jadi Garda Terdepan Perkuat Nila...,news,2026-11-07T21:45:00+07:00,https://news.detik.com/berita/d-8743059/mpr-aj...
3,Istri di Jakbar Dianiaya hingga Disundut Rokok...,news,2026-11-07T21:42:00+07:00,https://news.detik.com/berita/d-8743055/istri-...
4,Damkar Evakuasi Cincin Tersangkut di Kemaluan ...,news,2026-11-07T20:47:00+07:00,https://news.detik.com/berita/d-8742997/damkar...
5,"Seskab Teddy Pamer Fasilitas Sekolah Rakyat, A...",news,2026-11-07T20:31:00+07:00,https://news.detik.com/berita/d-8742990/seskab...
6,Om Tikam Keponakan hingga Kritis di Maros Usai...,news,2026-11-07T20:27:00+07:00,https://news.detik.com/berita/d-8742985/om-tik...
14,Apa Dampak Kemenangan Demokrat di Pemilu Sela ...,news,2026-11-07T17:40:00+07:00,https://news.detik.com/bbc-world/d-8742802/apa...
15,"Kalahkan Republik dalam Pemilu Sela AS, Demokr...",news,2026-11-04T18:07:00+07:00,https://news.detik.com/bbc-world/d-8738485/kal...
16,Menhub Spanyol Mundur Buntut Kecelakaan Kereta...,news,2026-11-04T17:51:00+07:00,https://news.detik.com/bbc-world/d-8738440/men...


In [46]:
print("Kategori yang muncul:")

print(
    df_latihan["kategori"]
    .value_counts(dropna=False)
)

Kategori yang muncul:
kategori
news    13
         7
Name: count, dtype: int64


Berdasarkan hasil crawling terhadap tiga sitemap, yaitu berita, inet, dan bbc-world, diperoleh sebanyak 20 judul berita. Kategori yang berhasil muncul adalah news sebanyak 13 data, sedangkan 7 data lainnya tidak memiliki kategori karena URL artikel kosong.

Setelah diterapkan aturan kualitas dari H-6, diperoleh hasil bahwa judul berhasil lolos seluruh aturan, sedangkan 7 judul gagal. Kegagalan tersebut disebabkan oleh kategori dan URL artikel yang kosong. Sementara itu, seluruh judul sudah terisi, memiliki panjang 3-30 kata, tanggal valid, dan tidak ditemukan judul duplikat.

Dengan demikian, sebanyak 65% data berhasil memenuhi seluruh aturan kualitas, sedangkan 35% lainnya belum memenuhi aturan. Hal ini menunjukkan bahwa tidak semua sitemap menghasilkan data yang lengkap, sehingga perlu dilakukan pemeriksaan terhadap URL sebelum data digunakan lebih lanjut.

7. Kata kunci. Kolom kata_kunci berisi kata kunci tiap berita (dipisah koma). Tulis mapper dan reducer untuk menghitung kata kunci terbanyak pada hasil crawl Anda.

In [47]:
import collections

def mapper_keyword(df):
    hasil = []

    for kata_kunci in df["kata_kunci"].dropna():
        daftar_keyword = kata_kunci.split(",")

        for kata in daftar_keyword:
            kata = kata.strip().lower()

            if kata:
                hasil.append((kata, 1))

    return hasil

In [48]:
def reducer_keyword(data_mapper):
    hasil = collections.Counter()

    for kata, jumlah in data_mapper:
        hasil[kata] += jumlah

    return hasil

In [49]:
# proses mapper
hasil_mapper = mapper_keyword(df_latihan)

print("Jumlah pasangan hasil mapper:", len(hasil_mapper))
print("\nContoh hasil mapper:")
print(hasil_mapper[:10])


# proses reducer
hasil_reducer = reducer_keyword(hasil_mapper)

print("\nJumlah kata kunci unik:", len(hasil_reducer))
print("\n10 kata kunci terbanyak:")

for kata, jumlah in hasil_reducer.most_common(10):
    print(f"{kata:<25} : {jumlah}")

Jumlah pasangan hasil mapper: 76

Contoh hasil mapper:
[('tramadol', 1), ('polres bogor', 1), ('bogor', 1), ('kebakaran', 1), ('stasiun gambir', 1), ('jakarta pusat', 1), ('damkar', 1), ('kebakaran stasiun gambir', 1), ('tenant', 1), ('bnr', 1)]

Jumlah kata kunci unik: 56

10 kata kunci terbanyak:
bbc world                 : 6
bbc                       : 6
amerika serikat           : 4
pemilu sela as            : 3
damkar                    : 2
bnr                       : 2
operator                  : 2
telkomsel                 : 2
lipsus smartfren          : 2
tramadol                  : 1


In [50]:
tabel_keyword = pd.DataFrame(
    hasil_reducer.most_common(10),
    columns=["Kata Kunci", "Jumlah Kemunculan"]
)

display(tabel_keyword)

,Kata Kunci,Jumlah Kemunculan
0,bbc world,6
1,bbc,6
2,amerika serikat,4
3,pemilu sela as,3
4,damkar,2
5,bnr,2
6,operator,2
7,telkomsel,2
8,lipsus smartfren,2
9,tramadol,1


Berdasarkan hasil proses Mapper dan Reducer, kata kunci yang paling sering muncul adalah 'bbc-world' dan 'bbc', masing-masing sebanyak 6 kali. Selanjutnya, kata kunci 'amerika serikat' muncul sebanyak 4 kali dan 'pemilu sela as' sebanyak 3 kali.

Hasil tersebut menunjukkan bahwa berita dari bagian BBC World cukup mendominasi kata kunci pada data crawling. Selain itu, terdapat kata kunci seperti 'damkar', 'bogor', dan 'telkomsel' yang masing-masing muncul sebanyak 2 kali.

Mapper berfungsi memisahkan kata kunci dari setiap berita dan memberikan nilai 1 untuk setiap kemunculan. Selanjutnya, Reducer menjumlahkankata kunci yang sama sehingga diperoleh daftar kata kunci yang paling sering muncul.

# Tugas Mandiri

In [51]:
df_index = bersih.copy()

print(df_index.columns.tolist())
print("Jumlah judul:", len(df_index))

['judul', 'url', 'kategori', 'tanggal', 'bulan', 'id']
Jumlah judul: 90818


In [52]:
# Ubah ID menjadi tipe string agar bisa digabung dengan ID buatan
df_index["id"] = df_index["id"].astype("string")

# Isi ID yang kosong dengan ID buatan berdasarkan kategori dan nomor baris
mask = df_index["id"].isna() | (df_index["id"].str.strip() == "")

df_index.loc[mask, "id"] = [
    f"{kategori}_baris_{i:06d}"
    for i, kategori in zip(
        df_index.index[mask],
        df_index.loc[mask, "kategori"].fillna("unknown")
    )
]

print("Jumlah ID kosong setelah diperbaiki:", df_index["id"].isna().sum())
print("Contoh ID pengganti:")
display(df_index.loc[mask, ["judul", "kategori", "id"]].head())

Jumlah ID kosong setelah diperbaiki: 0
Contoh ID pengganti:


,judul,kategori,id
39732,Google Sampai Kekurangan Laptop Karena Work Fr...,inet,inet_baris_039732
39733,APU AMD Renoir Bakal Punya 8 Core?,inet,inet_baris_039733
39734,Peran Telkom Atasi COVID-19: Sediakan Layanan ...,inet,inet_baris_039734
39735,"Viral Surat Bebas COVID-19 Dijual Online, Ini ...",inet,inet_baris_039735
39736,"Data Mitra GoFood Aman, Pakar IT: Waspadai Kej...",inet,inet_baris_039736


In [53]:
def mapper_inverted_index(df):
    hasil = []

    for _, row in df.iterrows():
        id_judul = row["id"]
        judul = row["judul"]

        kata_kata = tokenisasi(judul)

        for kata in set(kata_kata):
            hasil.append((kata, id_judul))

    return hasil


hasil_mapper_index = mapper_inverted_index(df_index)

print("Jumlah pasangan hasil mapper:", len(hasil_mapper_index))
print("Contoh hasil mapper:")
print(hasil_mapper_index[:15])

Jumlah pasangan hasil mapper: 817993
Contoh hasil mapper:
[('awasi', '4916114'), ('meikarta', '4916114'), ('kemnaker', '4916114'), ('tka', '4916114'), ('di', '4916114'), ('bni', '4916199'), ('digitalkan', '4916199'), ('java', '4916199'), ('jazz', '4916199'), ('ke', '4916193'), ('mau', '4916193'), ('edhy', '4916193'), ('genjot', '4916193'), ('australia', '4916193'), ('daya', '4916193')]


In [54]:
from collections import defaultdict

def reducer_inverted_index(data_mapper):
    hasil = defaultdict(set)

    for kata, id_judul in data_mapper:
        hasil[kata].add(id_judul)

    return {
        kata: sorted(daftar_id)
        for kata, daftar_id in hasil.items()
    }


hasil_index = reducer_inverted_index(hasil_mapper_index)

print("Jumlah kata unik:", len(hasil_index))

Jumlah kata unik: 32275


In [55]:
# Urutkan kata berdasarkan jumlah judul yang memuatnya
kata_teratas = sorted(
    hasil_index.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:3]

hasil_contoh = []

for kata, daftar_id in kata_teratas:
    hasil_contoh.append({
        "Kata": kata,
        "Jumlah Judul": len(daftar_id),
        "Lima ID Pertama": daftar_id[:5]
    })

tabel_index = pd.DataFrame(hasil_contoh)

display(tabel_index)

,Kata,Jumlah Judul,Lima ID Pertama
0,di,26180,"[4841806, 4841929, 4841931, 4841932, 4841936]"
1,corona,17228,"[4856495, 4856598, 4868317, 4868545, 4869163]"
2,ini,10085,"[4841859, 4841927, 4841929, 4841930, 4841936]"


In [56]:
import json

with open("inverted_index.json", "w", encoding="utf-8") as f:
    json.dump(
        hasil_index,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Inverted index berhasil disimpan!")

Inverted index berhasil disimpan!


Berdasarkan proses MapReduce yang dilakukan, inverted index berhasil dibuat dan disimpan. Inverted index merupakan struktur data yang digunakan untuk mencatat daftar ID judul yang mengandung suatu kata.

Hasil ini menunjukkan bahwa kata 'di' muncul pada 26.180 judul, kata 'corona' muncul pada 17.228 judul, dan kata 'ini' muncul pada 10.085 judul.

Sebagai contoh, kata 'corona' memiliki lima ID pertama, yaitu 4856495, 4856598, 4868317, 4868545, dan 4869163. Dengan demikian, pengguna dapat mengetahui judul mana saja yang mengandung kata tertentu melalui daftar ID yang tersimpan.

Untuk kategori inet, kolom ID yang kosong perlu ditangani agar tidak menghasilkan ID yang tidak valid. Penanganan yang dapat digunakan adalah memberikan ID pengganti yang unik untuk setiap judul inet. Cara ini memungkinkan seluruh judul tetap masuk ke dalam inverted index tanpa mencampuradukkan identitas judul.